# Plant Disease Detection — EDA & Training

This notebook walks through:
1. **EDA** of the PlantVillage dataset (class distribution, sample images).
2. **Training** the MobileNetV2 transfer-learning model.
3. **Evaluation** with a confusion matrix and per-class metrics.

> Run `python download_dataset.py` first (or place images at `data/PlantVillage/<class>/*.jpg`).

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Make the project importable from a notebook
sys.path.insert(0, os.path.abspath('..'))
import config

sns.set_style('whitegrid')

## 1. Class distribution

In [ ]:
data_dir = Path(config.DATA_DIR)
counts = {p.name: len(list(p.glob('*.jpg')) + list(p.glob('*.JPG')) + list(p.glob('*.png')))
          for p in data_dir.iterdir() if p.is_dir()}
df = pd.DataFrame({'class': list(counts.keys()), 'n_images': list(counts.values())})
df = df.sort_values('n_images', ascending=True)
print(f'Total classes: {len(df)}')
print(f'Total images : {df.n_images.sum():,}')
print(f'Min / max per class: {df.n_images.min()} / {df.n_images.max()}')

plt.figure(figsize=(8, 12))
plt.barh(df['class'], df['n_images'], color='#22c55e')
plt.title('Images per class (PlantVillage)')
plt.xlabel('Count')
plt.tight_layout()
plt.show()

## 2. Sample images

In [ ]:
from PIL import Image

sample_classes = df.sample(12, random_state=42)['class'].tolist()
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for ax, cls in zip(axes.flat, sample_classes):
    files = list((data_dir / cls).glob('*.jpg'))
    if not files: files = list((data_dir / cls).glob('*.JPG'))
    img = Image.open(files[0])
    ax.imshow(img)
    ax.set_title(config.pretty_name(cls), fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Train the model

For a full run, use the CLI: `python train.py --epochs 20 --model transfer`.

Below is a short 3-epoch smoke test:

In [ ]:
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from model import build_transfer_model

gen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    rotation_range=30, horizontal_flip=True,
    validation_split=0.2,
)
train = gen.flow_from_directory(str(data_dir), target_size=(config.IMG_SIZE, config.IMG_SIZE),
                                 batch_size=32, subset='training', class_mode='categorical', seed=42)
val = gen.flow_from_directory(str(data_dir), target_size=(config.IMG_SIZE, config.IMG_SIZE),
                               batch_size=32, subset='validation', class_mode='categorical', seed=42)

model, base = build_transfer_model(trainable_backbone=False)
model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
history = model.fit(train, validation_data=val, epochs=3)

## 4. Plot training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'], label='train')
ax1.plot(history.history['val_accuracy'], label='val')
ax1.set_title('Accuracy'); ax1.legend()
ax2.plot(history.history['loss'], label='train')
ax2.plot(history.history['val_loss'], label='val')
ax2.set_title('Loss'); ax2.legend()
plt.tight_layout(); plt.show()

## 5. Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

val.reset()
y_pred = model.predict(val, verbose=1).argmax(axis=1)
y_true = val.classes[:len(y_pred)]
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, cmap='Greens', xticklabels=False, yticklabels=False, cbar=True)
plt.title('Confusion matrix — 38 classes')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.show()

print(classification_report(y_true, y_pred, target_names=config.CLASS_NAMES, zero_division=0))